# Chest X-Ray EDA
Exploratory Data Analysis — understand the dataset before writing any model code.
Rule: never train a model on data you haven't looked at.

In [ ]:
import sys
sys.path.append('..')  # so we can import from src/

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image
import torch

DATA_DIR = '../data/raw/chest_xray'
print('Setup complete.')

## 1. Dataset Statistics

In [ ]:
from src.data.dataset import ChestXrayDataset
from src.data.transforms import get_train_transforms, get_val_transforms

train_ds = ChestXrayDataset(f'{DATA_DIR}/train')
val_ds   = ChestXrayDataset(f'{DATA_DIR}/val')
test_ds  = ChestXrayDataset(f'{DATA_DIR}/test')

print('=== SPLIT STATISTICS ===')
for name, ds in [('TRAIN', train_ds), ('VAL', val_ds), ('TEST', test_ds)]:
    print(f'\n--- {name} ---')
    ds.summary()

## 2. Class Distribution Bar Chart
Visualizing class imbalance — this will inform our loss function choice.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
splits = [('Train', train_ds), ('Val', val_ds), ('Test', test_ds)]
colors = ['#2ecc71', '#e74c3c']

for ax, (name, ds) in zip(axes, splits):
    counts = [ds.class_counts[0], ds.class_counts[1]]
    bars = ax.bar(['NORMAL', 'PNEUMONIA'], counts, color=colors, edgecolor='black', width=0.5)
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                str(count), ha='center', va='bottom', fontweight='bold')
    ax.set_title(f'{name} Split', fontsize=14, fontweight='bold')
    ax.set_ylabel('Image Count')
    ax.set_ylim(0, max(counts) * 1.15)

plt.suptitle('Class Distribution Across Splits', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: data/processed/class_distribution.png')

## 3. Sample Images — Normal vs Pneumonia
Always look at your data. What does a pneumonia X-ray actually look like?

In [ ]:
def show_samples(data_dir, n=5):
    fig, axes = plt.subplots(2, n, figsize=(3*n, 7))
    fig.suptitle('Sample X-rays: NORMAL (top) vs PNEUMONIA (bottom)', fontsize=14, fontweight='bold')

    for col, cls in enumerate(['NORMAL', 'PNEUMONIA']):
        cls_dir = Path(data_dir) / 'train' / cls
        imgs = sorted(cls_dir.glob('*.jpeg'))[:n]
        row = 0 if cls == 'NORMAL' else 1
        for i, img_path in enumerate(imgs):
            img = np.array(Image.open(img_path).convert('L'))  # grayscale for display
            axes[row][i].imshow(img, cmap='gray')
            axes[row][i].set_title(cls, color='green' if cls=='NORMAL' else 'red', fontsize=9)
            axes[row][i].axis('off')

    plt.tight_layout()
    plt.savefig('../data/processed/sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()

show_samples(DATA_DIR)
print('Saved: data/processed/sample_images.png')

## 4. Image Size Distribution
Are all images the same size? Models need fixed-size input — this tells us how much resize distortion we'll introduce.

In [ ]:
widths, heights = [], []
train_dir = Path(DATA_DIR) / 'train'

for cls in ['NORMAL', 'PNEUMONIA']:
    for img_path in (train_dir / cls).glob('*.jpeg'):
        w, h = Image.open(img_path).size
        widths.append(w)
        heights.append(h)

print(f'Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}')
print(f'Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(widths, bins=30, color='steelblue', edgecolor='black')
ax1.set_title('Image Width Distribution'); ax1.set_xlabel('Pixels')
ax2.hist(heights, bins=30, color='coral', edgecolor='black')
ax2.set_title('Image Height Distribution'); ax2.set_xlabel('Pixels')
plt.tight_layout()
plt.savefig('../data/processed/image_size_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Pixel Intensity Distribution
What is the average brightness of NORMAL vs PNEUMONIA scans?
Pneumonia causes 'consolidation' — lung fills with fluid → brighter/whiter region on X-ray.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for cls, color in [('NORMAL', '#2ecc71'), ('PNEUMONIA', '#e74c3c')]:
    intensities = []
    for img_path in list((train_dir / cls).glob('*.jpeg'))[:200]:  # sample 200
        img = np.array(Image.open(img_path).convert('L').resize((224, 224)))
        intensities.append(img.mean())
    ax.hist(intensities, bins=40, alpha=0.6, color=color, label=cls, edgecolor='black')

ax.set_title('Mean Pixel Intensity: NORMAL vs PNEUMONIA', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Pixel Value (0=black, 255=white)')
ax.set_ylabel('Count')
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig('../data/processed/intensity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nKey insight: Pneumonia lungs tend to be brighter (fluid = white on X-ray)')

## 6. Augmentation Preview
Visualize what our training augmentations actually look like on a real X-ray.
This confirms we're not destroying anatomy with our transforms.

In [ ]:
import albumentations as A
from src.data.transforms import get_train_transforms

# Get one sample pneumonia image
sample_path = sorted((train_dir / 'PNEUMONIA').glob('*.jpeg'))[0]
original = np.array(Image.open(sample_path).convert('RGB'))

# Show original + 6 augmented versions
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Augmentation Preview (same image, 7 random augmentations)', fontsize=13, fontweight='bold')

# Build pipeline without normalize/ToTensor for display
aug_display = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(p=0.3),
])

axes[0][0].imshow(np.array(Image.open(sample_path).convert('RGB').resize((224,224))))
axes[0][0].set_title('ORIGINAL', fontweight='bold')
axes[0][0].axis('off')

for idx, ax in enumerate(axes.flat[1:]):
    augmented = aug_display(image=original)['image']
    ax.imshow(augmented)
    ax.set_title(f'Augmented #{idx+1}')
    ax.axis('off')

plt.tight_layout()
plt.savefig('../data/processed/augmentation_preview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: data/processed/augmentation_preview.png')

## 7. DataLoader Batch Test
Final check: load one batch through the full pipeline and verify shapes + types are correct.

In [ ]:
from src.data.dataloader import get_dataloaders

train_loader, val_loader, test_loader, train_ds = get_dataloaders(
    data_dir=DATA_DIR,
    image_size=224,
    batch_size=32,
    num_workers=0,
)

images, labels = next(iter(train_loader))

print('=== BATCH VERIFICATION ===')
print(f'images.shape : {images.shape}')   # expect [32, 3, 224, 224]
print(f'images.dtype : {images.dtype}')   # expect torch.float32
print(f'labels.shape : {labels.shape}')   # expect [32]
print(f'labels unique: {labels.unique()}') # expect tensor([0, 1])
print(f'pixel range  : [{images.min():.2f}, {images.max():.2f}]')  # normalized range
print(f'class weights: {train_ds.get_class_weights()}')
print(f'\ntrain batches: {len(train_loader)}')
print(f'val batches  : {len(val_loader)}')
print(f'test batches : {len(test_loader)}')
print('\nData pipeline is working correctly.')